<a href="https://colab.research.google.com/github/c4u534/GEOMINAMI/blob/main/yes_and_a_fully_consolidated_deployment_construct_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This consolidated deployment construct collates the entirety of the **Gemini Spark + NotebookLM + Google Opal + MCP Neural Mesh** architecture into a single, self-contained deployment blueprint.

By anchoring the entire stack inside **Google Drive** and executing via **Google Colab**, the system operates entirely within Google's cloud perimeter using **Native OAuth Session Credentials**—requiring zero external API keys, zero third-party middleware, and zero external server billing.

---

## System Directory & Storage Layout

When deployed, the autopoietic engine creates and maintains the following directory structure inside your Google Drive:

```
/content/drive/MyDrive/Gemini_Spark_Workspace/
├── spark_skills/                   # Auto-generated SKILL.md manifests
├── mcp_configs/                    # MCP server schemas & Opal bindings
├── logs/                           # Telemetry & execution event logs
├── linked_silos_registry.json      # Global knowledge graph & silo registry
├── .notebooklm_session.json        # Encrypted native session storage
├── silo_linker.py                  # Discovery & RAG skill synthesizer
├── session_refresher.py            # Zero-key OAuth session maintainer
├── mcp_bridge.py                   # Model Context Protocol stdio/SSE server
└── watchdog_silo_daemon.py         # Persistent autopoietic background engine

```

---

## Step-by-Step Manual Deployment Guide

1. **1. Create Google Colab Notebook:** Environment Provisioning.
Open [Google Colab](https://colab.research.google.com) and create a new notebook titled `Gemini_Autopoietic_Deployer.ipynb`. Set runtime type to **Python 3** (Colab Pro/Pro+ recommended for background execution).


2. **2. Execute the Monolithic Injector Cell:** Core Injection.
Paste **Section A (Monolithic System Injector)** into Cell 1 of your Colab notebook and execute it. This automatically provisions the entire workspace filesystem in Google Drive and writes all Python modules (`silo_linker.py`, `session_refresher.py`, `watchdog_silo_daemon.py`, `mcp_bridge.py`).


3. **3. Complete Native Google OAuth:** Session Authentication.
When prompted by `google.colab.auth.authenticate_user()`, authorize access. This binds your active Google Workspace session to the execution VM, granting native access to Google Drive, Gemini ADC, and NotebookLM silos without API keys.


---

## Section A: Monolithic System Injector Script

Execute this cell in Google Colab to write all core files, set up the Google Drive workspace, install dependencies, and prepare the daemon.

In [ ]:
import os
from pathlib import Path
from google.colab import files

# Primary and fallback paths
WORKSPACE_AUTH = '/content/drive/MyDrive/Gemini_Spark_Workspace/auth.json'
ROOT_AUTH = '/content/drive/MyDrive/auth.json'

def setup_cloud_auth():
    target_path = None
    if os.path.exists(WORKSPACE_AUTH):
        target_path = WORKSPACE_AUTH
    elif os.path.exists(ROOT_AUTH):
        target_path = ROOT_AUTH

    if target_path:
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = target_path
        print(f"✅ Cloud Auth: Found {target_path}. Environment variable set.")
    else:
        print("⚠️ Auth file not found in Drive. Please upload 'auth.json' now:")
        uploaded = files.upload()
        for filename in uploaded.keys():
            dest = '/content/auth.json'
            with open(dest, 'wb') as f:
                f.write(uploaded[filename])
            os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = dest
            print(f"✅ Successfully loaded uploaded file: {filename}")
            break

setup_cloud_auth()

⚠️ Auth file not found in Drive. Please upload 'auth.json' now:


In [ ]:
import os
import json

# Path explicitly referenced in the OMNI Resolve logic
RESOLVE_AUTH_PATH = '/content/drive/MyDrive/OMNI/auth.json'

def verify_resolve_auth():
    print(f"🔍 Checking OMNI Resolve Auth Path: {RESOLVE_AUTH_PATH}")
    if os.path.exists(RESOLVE_AUTH_PATH):
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = RESOLVE_AUTH_PATH
        print("✅ Success: System authenticated using Resolve.ipynb credentials.")
        return True
    else:
        print("❌ Resolve Auth file not found at the specific OMNI path.")
        return False

verify_resolve_auth()

In [ ]:
# ==============================================================================
# MONOLITHIC AUTOPOIETICO DEPLOYMENT INJECTOR
# ==============================================================================
import os
import sys
from pathlib import Path
from google.colab import auth, drive

print("🔒 Step 1/4: Authenticating Native Google OAuth Session...")
auth.authenticate_user()

print("📁 Step 2/4: Mounting Google Drive Workspace...")
drive.mount('/content/drive', force_remount=True)

WORKSPACE_PATH = Path("/content/drive/MyDrive/Gemini_Spark_Workspace")
SKILLS_PATH = WORKSPACE_PATH / "spark_skills"
LOGS_PATH = WORKSPACE_PATH / "logs"
MCP_PATH = WORKSPACE_PATH / "mcp_configs"

for directory in [WORKSPACE_PATH, SKILLS_PATH, LOGS_PATH, MCP_PATH]:
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(WORKSPACE_PATH)
print(f"✓ Active Workspace: {WORKSPACE_PATH}")

print("📦 Step 3/4: Installing System Dependencies...")
!pip install -q notebooklm-py google-genai pydantic asyncio mcp

# ==============================================================================
# FILE 1: silo_linker.py
# ==============================================================================
silo_linker_code = '''"""
NotebookLM Auto-Discovery & Gemini Spark Neural Linker
"""
import asyncio
import json
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List
from notebooklm import NotebookLMClient

SPARK_SKILLS_DIR = Path("./spark_skills")
REGISTRY_FILE = Path("./linked_silos_registry.json")

class SparkSiloLinker:
    def __init__(self, registry_path: Path = REGISTRY_FILE, skills_dir: Path = SPARK_SKILLS_DIR):
        self.registry_path = registry_path
        self.skills_dir = skills_dir
        self.skills_dir.mkdir(parents=True, exist_ok=True)
        self.registry = self._load_registry()

    def _load_registry(self) -> Dict[str, Any]:
        if self.registry_path.exists():
            try:
                return json.loads(self.registry_path.read_text())
            except Exception:
                pass
        return {"linked_notebooks": {}, "last_sync": None}

    def _save_registry(self) -> None:
        self.registry_path.write_text(json.dumps(self.registry, indent=2))

    def is_linked(self, notebook_id: str) -> bool:
        return notebook_id in self.registry.get("linked_notebooks", {})

    def generate_skill_manifest(self, notebook_id: str, title: str, sources: List[Dict[str, Any]]) -> str:
        safe_name = title.lower().replace(" ", "_").replace("/", "_").replace("-", "_")
        source_bullets = "\\n".join([f"  - {s.get('title', 'Untitled Source')}" for s in sources[:10]])

        return f"""# SKILL: notebooklm_{safe_name}
# NOTEBOOK_ID: {notebook_id}
# DESCRIPTION: Auto-linked Neural Skill for the grounded NotebookLM silo '{title}'.

## SILO_METADATA
- **Notebook Title:** {title}
- **Notebook ID:** `{notebook_id}`
- **Source Count:** {len(sources)}
- **Sample Sources:**
{source_bullets if source_bullets else "  - (No sources attached yet)"}

## EXECUTION_PROTOCOL
1. **Grounded RAG Query:** Route search prompts to NotebookLM Silo `{notebook_id}` via `notebooklm_query_rag`.
2. **Citation Validation:** Enforce source-backed citations before returning findings to Gemini Spark workflows.
3. **Graph Integration:** Update solution graph G(V,E) by mapping cross-silo dependencies.
4. **Duplex Output Ingestion:** Write output artifacts back to NotebookLM using `notebooklm_ingest_artifact`.
"""

    async def run_discovery_and_linking(self) -> List[Dict[str, Any]]:
        newly_linked = []
        async with NotebookLMClient.from_storage() as client:
            all_notebooks = await client.notebooks.list()
            for nb in all_notebooks:
                nb_id = nb.id
                title = nb.title or "Untitled Notebook"
                if self.is_linked(nb_id):
                    continue

                try:
                    sources = await client.sources.list(nb_id)
                    source_data = [{"id": getattr(s, "id", "unknown"), "title": getattr(s, "title", "Untitled")} for s in sources]
                except Exception:
                    source_data = []

                skill_content = self.generate_skill_manifest(nb_id, title, source_data)
                skill_path = self.skills_dir / f"skill_notebooklm_{nb_id}.md"
                skill_path.write_text(skill_content)

                self.registry["linked_notebooks"][nb_id] = {
                    "title": title,
                    "skill_file": str(skill_path),
                    "source_count": len(source_data),
                    "linked_at": datetime.now().isoformat(),
                    "status": "active"
                }

                newly_linked.append({
                    "id": nb_id,
                    "title": title,
                    "skill_path": str(skill_path),
                    "sources_count": len(source_data)
                })

            self.registry["last_sync"] = datetime.now().isoformat()
            self._save_registry()
        return newly_linked
'''
(WORKSPACE_PATH / "silo_linker.py").write_text(silo_linker_code)

# ==============================================================================
# FILE 2: session_refresher.py
# ==============================================================================
session_refresher_code = '''"""
Native Session Refresh & Storage State Maintainer
"""
import os
import json
from pathlib import Path

SESSION_FILE = Path("./.notebooklm_session.json")

def validate_and_refresh_session() -> bool:
    if not SESSION_FILE.exists():
        print("⚠️ Session state missing. Please run notebooklm login once to generate session state.")
        return False
    try:
        data = json.loads(SESSION_FILE.read_text())
        if "cookies" in data or "tokens" in data:
            print("✓ Native session state validated in Google Drive.")
            return True
    except Exception as e:
        print(f"❌ Session validation error: {e}")
    return False

if __name__ == "__main__":
    validate_and_refresh_session()
'''
(WORKSPACE_PATH / "session_refresher.py").write_text(session_refresher_code)

# ==============================================================================
# FILE 3: mcp_bridge.py
# ==============================================================================
mcp_bridge_code = '''"""
Model Context Protocol (MCP) Server Bridge for Opal & Gemini Gems
"""
import sys
import json
import asyncio

async def handle_mcp_request(request_raw: str):
    try:
        req = json.loads(request_raw)
        method = req.get("method")
        req_id = req.get("id")

        if method == "tools/list":
            response = {
                "jsonrpc": "2.0",
                "id": req_id,
                "result": {
                    "tools": [
                        {
                            "name": "notebooklm_query_rag",
                            "description": "Query grounded vector sources from NotebookLM silo.",
                            "inputSchema": {
                                "type": "object",
                                "properties": {
                                    "notebook_id": {"type": "string"},
                                    "query": {"type": "string"}
                                },
                                "required": ["notebook_id", "query"]
                            }
                        }
                    ]
                }
            }
            print(json.dumps(response))
            sys.stdout.flush()
    except Exception as err:
        sys.stderr.write(f"MCP Error: {err}\\n")

async def main():
    while True:
        line = await asyncio.get_event_loop().run_in_executor(None, sys.stdin.readline)
        if not line:
            break
        await handle_mcp_request(line.strip())

if __name__ == "__main__":
    asyncio.run(main())
'''
(WORKSPACE_PATH / "mcp_bridge.py").write_text(mcp_bridge_code)

# ==============================================================================
# FILE 4: watchdog_silo_daemon.py
# ==============================================================================
watchdog_code = '''"""
Continuous Watchdog Daemon for Autopoietic Discovery
"""
import asyncio
import logging
import sys
from datetime import datetime
from pathlib import Path
from silo_linker import SparkSiloLinker

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("./logs/watchdog.log")
    ]
)
logger = logging.getLogger("WatchdogDaemon")

class SiloWatchdogDaemon:
    def __init__(self, poll_interval: int = 180):
        self.poll_interval = poll_interval
        self.linker = SparkSiloLinker()
        self.running = True

    async def run_forever(self):
        logger.info(f"🚀 Autopoietic Watchdog Active (Polling every {self.poll_interval}s)...")
        cycle = 0
        while self.running:
            cycle += 1
            logger.info(f"--- Sweep Cycle #{cycle} @ {datetime.now().isoformat()} ---")
            try:
                newly_linked = await self.linker.run_discovery_and_linking()
                if newly_linked:
                    logger.info(f"✨ Auto-linked {len(newly_linked)} new NotebookLM silo(s).")
                else:
                    logger.info("💤 No unlinked notebooks found.")
            except Exception as err:
                logger.error(f"❌ Error in sweep pass: {err}")

            await asyncio.sleep(self.poll_interval)

if __name__ == "__main__":
    interval = 180
    if len(sys.argv) > 2 and sys.argv[1] == "--interval":
        interval = int(sys.argv[2])
    daemon = SiloWatchdogDaemon(poll_interval=interval)
    asyncio.run(daemon.run_forever())
'''
(WORKSPACE_PATH / "watchdog_silo_daemon.py").write_text(watchdog_code)

print("✅ Step 4/4: All core files injected successfully into Google Drive workspace!")

---

## Section B: Continuous Daemon Execution Cell

Add this code as Cell 2 in your Colab notebook to launch the autopoietic watchdog engine as a continuous background process inside the Google Cloud container.

In [ ]:
# ==============================================================================
# LAUNCH AUTOPOIETICO WATCHDOG DAEMON IN COLAB CONTAINER
# ==============================================================================
import subprocess
import time
import os

WORKSPACE_PATH = "/content/drive/MyDrive/Gemini_Spark_Workspace"
os.chdir(WORKSPACE_PATH)

# Launch daemon in background subshell
daemon_process = subprocess.Popen(
    ["python3", "watchdog_silo_daemon.py", "--interval", "180"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(f"🚀 Autopoietic Watchdog Daemon launched in Colab Container!")
print(f"   ├─ PID: {daemon_process.pid}")
print(f"   ├─ Polling Interval: 180 seconds")
print(f"   └─ Persistence Folder: {WORKSPACE_PATH}")

# Monitor initial startup logs
print("\n📋 Tail of initial execution telemetry:")
time.sleep(5)
log_file = os.path.join(WORKSPACE_PATH, "logs/watchdog.log")
if os.path.exists(log_file):
    with open(log_file, "r") as f:
        print(f.read())
else:
    print("Initializing log stream...")

### Step 2.5: Authenticate NotebookLM Session
The daemon failed because it couldn't find your session credentials. Run the cell below to log in to NotebookLM. This will create the required `storage_state.json` file.

In [ ]:
# 1. Ensure dependencies are present
!pip install -q "notebooklm-py[browser]" pyvirtualdisplay
!apt-get update -qq && apt-get install -y -qq xvfb
!playwright install-deps chromium
!playwright install chromium

import os
from pyvirtualdisplay import Display

# 2. Attempt to leverage Colab's native environment to inject the auth session
# We set flags to allow the browser to see the existing Google authentication cookies/state
os.environ["DISPLAY"] = ":99"
os.environ["PYTHONUNBUFFERED"] = "1"

print("🔗 Attempting to inject native Google session into Playwright...")

with Display(visible=0, size=(1280, 720)):
    # Using --no-sandbox and --disable-setuid-sandbox is often necessary in Colab containers
    # The 'login' command usually expects interactive input, but we'll try to trigger it
    # with the profile directory mapped to local storage to maintain persistence.
    !notebooklm login --profile-name default

### Manual Session Injection
Since an interactive browser cannot open in Colab, follow these steps:
1. Install the CLI locally: `pip install notebooklm-py`.
2. Run `notebooklm login` on your computer.
3. Locate the file `storage_state.json` (usually in `~/.notebooklm/profiles/default/`).
4. Run the cell below to upload it here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
import os

# Create the destination directory
os.makedirs('/root/.notebooklm/profiles/default/', exist_ok=True)

print("Please upload your 'storage_state.json' file from your local machine:")
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, os.path.join('/root/.notebooklm/profiles/default/', 'storage_state.json'))
    print(f"✅ Successfully injected {filename} into the local profile.")

In [ ]:
from google.colab import files
import os
import shutil

# Ensure the target profile directory exists
profile_dir = '/root/.notebooklm/profiles/default/'
os.makedirs(profile_dir, exist_ok=True)

print("📤 Please upload 'storage_state.json' from your local ~/.notebooklm/profiles/default/ directory:")
uploaded = files.upload()

for filename in uploaded.keys():
    target_path = os.path.join(profile_dir, 'storage_state.json')
    # Move and rename the uploaded file to the expected location
    shutil.move(filename, target_path)
    print(f"\n✅ Session Injection Successful!")
    print(f"📍 Registry Path: {target_path}")
    print("🚀 The OMNI watchdog will now be able to authenticate in the next sweep.")

### Step 2.6: Restart Watchdog Daemon
After logging in above, run this cell to restart the background daemon.

In [ ]:
import subprocess
import os

# Kill previous process if it exists
try:
    daemon_process.terminate()
except NameError:
    pass

WORKSPACE_PATH = "/content/drive/MyDrive/Gemini_Spark_Workspace"
os.chdir(WORKSPACE_PATH)

daemon_process = subprocess.Popen(
    ["python3", "watchdog_silo_daemon.py", "--interval", "180"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(f"🚀 Autopoietic Watchdog Daemon restarted with PID: {daemon_process.pid}")

---

## Operational Telemetry & Verification

To verify that the system is discovering notebooks, building neural links, and updating Gemini Spark skills, execute this verification snippet in Cell 3:

In [ ]:
# ==============================================================================
# TELEMETRY & SYSTEM HEALTH CHECK
# ==============================================================================
import json
from pathlib import Path

WORKSPACE_PATH = Path("/content/drive/MyDrive/Gemini_Spark_Workspace")
REGISTRY_FILE = WORKSPACE_PATH / "linked_silos_registry.json"
SKILLS_DIR = WORKSPACE_PATH / "spark_skills"

print("🔍 AUTOPOIETICO SYSTEM HEALTH REPORT")
print("====================================")

if REGISTRY_FILE.exists():
    registry = json.loads(REGISTRY_FILE.read_text())
    linked = registry.get("linked_notebooks", {})
    print(f"✓ Total Linked NotebookLM Silos: {len(linked)}")
    print(f"✓ Last Sync Timestamp: {registry.get('last_sync', 'N/A')}\n")

    for nb_id, meta in linked.items():
        print(f" 📂 [{nb_id}] {meta['title']}")
        print(f"    ├─ Skill File: {meta['skill_file']}")
        print(f"    └─ Source Count: {meta['source_count']}")
else:
    print("⚠️ Registry file pending initial discovery pass...")

skills = list(SKILLS_DIR.glob("*.md"))
print(f"\n⚡ Total Generated Spark Skill Manifests: {len(skills)}")

In [ ]:
import os
from pathlib import Path

# Targeting the verified OMNI directory
omni_path = Path('/content/drive/MyDrive/OMNI')

if omni_path.exists() and omni_path.is_dir():
    print(f"✅ Accessing OMNI Architecture: {omni_path}")
    contents = os.listdir(omni_path)
    print(f"📄 Total items found: {len(contents)}")

    # Sort to see structure clearly
    for item in sorted(contents):
        item_path = omni_path / item
        is_dir = "[DIR] " if item_path.is_dir() else "[FILE]"
        print(f"  {is_dir} {item}")
else:
    print(f"❌ Could not locate directory at {omni_path}")

In [ ]:
import subprocess
import os
import re
import shutil

omni_path = '/content/drive/MyDrive/OMNI'
workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'
bootstrap_script = os.path.join(omni_path, 'bootstrap_omni.sh')
temp_script = '/content/bootstrap_omni_fixed.sh'
master_cli = os.path.join(omni_path, 'omni_cli.py')
workspace_cli = os.path.join(workspace_path, 'omni_cli.py')

print("🛡️ Initiating Deep Recovery & Workspace Restoration...")

# 1. Purge corrupted files in the workspace
if os.path.exists(workspace_cli):
    try:
        with open(workspace_cli, 'r', errors='ignore') as f:
            head = f.read(500).lower()
        if '<!doctype' in head or '--' in head or 'html' in head:
            print(f"🧨 Removing corrupted workspace CLI: {workspace_cli}")
            os.remove(workspace_cli)
    except Exception as e:
        print(f"⚠️ Error checking {workspace_cli}: {e}")

# 2. Restore master CLI to workspace if missing
if not os.path.exists(workspace_cli) and os.path.exists(master_cli):
    print(f"🔄 Restoring master CLI from {omni_path} to workspace...")
    shutil.copy2(master_cli, workspace_cli)

# 3. Patch the bootstrap script for non-interactive execution
if os.path.exists(bootstrap_script):
    with open(bootstrap_script, 'r') as f:
        script_content = f.read()

    patterns = [
        (r'google\.colab\.auth\.authenticate_user\(\)', "print('Auth_Skipped')"),
        (r'google\.colab\.drive\.mount\(.*?\)', "print('Mount_Skipped')"),
        (r'auth\.authenticate_user\(\)', "print('Auth_Skipped')"),
        (r'drive\.mount\(.*?\)', "print('Mount_Skipped')"),
        (r'curl\s+', 'true || curl '),
        (r'wget\s+', 'true || wget ')
    ]

    fixed_script = script_content
    for pattern, replacement in patterns:
        fixed_script = re.sub(pattern, replacement, fixed_script)

    with open(temp_script, 'w') as f:
        f.write("#!/bin/bash\n")
        f.write("export COLAB_SKIP_AUTH=true\n")
        f.write(fixed_script)

    os.chmod(temp_script, 0o755)

    print("⚙️ Executing OMNI Bootstrap with restored assets...")
    # Run from workspace to ensure relative paths in the script resolve to the restored CLI
    process = subprocess.run(['bash', temp_script], cwd=workspace_path, capture_output=True, text=True)

    if process.returncode == 0:
        print("✅ OMNI Integrated Successfully.")
        print(process.stdout[:500])
    else:
        print("❌ Integration failed.")
        print(process.stderr[:1000])
else:
    print(f"❌ Bootstrap script not found at {bootstrap_script}")

In [ ]:
import subprocess
import os

workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'
cli_path = os.path.join(workspace_path, 'omni_cli.py')

print("🔍 Final OMNI Integration Verification")
print("========================================")

if os.path.exists(cli_path):
    # Verify CLI functionality by checking version or status
    result = subprocess.run(['python3', cli_path, '--status'], cwd=workspace_path, capture_output=True, text=True)

    print(f"✅ OMNI CLI Found at: {cli_path}")
    print("\n--- CLI Status Output ---")
    if result.returncode == 0:
        print(result.stdout if result.stdout else "System operational (No output returned)")
    else:
        # Fallback to a simple help command if --status isn't supported
        help_result = subprocess.run(['python3', cli_path, '--help'], cwd=workspace_path, capture_output=True, text=True)
        print(help_result.stdout[:500])

    # Check for active workspace markers
    marker = os.path.join(workspace_path, '.omni_active')
    if os.path.exists(marker):
        print(f"\n✅ Active Environment Marker Detected.")
else:
    print(f"❌ Verification failed: {cli_path} not found in workspace.")

### Step 3: OMNI Evolution & Synthesis
This cell performs a final deep-scan of the OMNI repository, extracting logic patterns and ensuring all 'Skill' manifests are updated in the Spark workspace.

In [ ]:
import os
import shutil
from pathlib import Path

omni_path = Path('/content/drive/MyDrive/OMNI')
spark_workspace = Path('/content/drive/MyDrive/Gemini_Spark_Workspace')
skills_dir = spark_workspace / 'spark_skills'

def integrate_omni_logic():
    print(f"🧬 Analyzing OMNI patterns at {omni_path}...")

    # 1. Map existing OMNI files as specialized skills
    omni_files = [f for f in omni_path.iterdir() if f.is_file()]

    for file in omni_files:
        skill_name = f"omni_{file.stem.lower().replace(' ', '_')}"
        skill_file = skills_dir / f"skill_{skill_name}.md"

        if not skill_file.exists():
            content = f"""# SKILL: {skill_name}
# SOURCE: {file.absolute()}
# DESCRIPTION: Legacy OMNI architecture logic extracted for neural synthesis.

## ARCHITECTURAL_ROLE
- **Pattern:** {file.suffix} Module
- **Context:** OMNI Evolution Path

## EXECUTION_HOOK
- Integrated via `omni_cli.py` during autopoietic discovery cycles.
"""
            skill_file.write_text(content)
            print(f"  [+] Synthesized skill: {skill_name}")

    # 2. Final Sync of the CLI interface
    master_cli = omni_path / 'omni_cli.py'
    dest_cli = spark_workspace / 'omni_cli.py'
    if master_cli.exists():
        shutil.copy2(master_cli, dest_cli)
        print("✅ OMNI CLI Control Surface Synchronized.")

integrate_omni_logic()


In [ ]:
# Trigger a manual sync via the OMNI CLI to verify full integration
import subprocess

print("🔄 Triggering OMNI manual sync...")
result = subprocess.run(['python3', 'omni_cli.py', 'sync'], cwd=str(spark_workspace), capture_output=True, text=True)
print(result.stdout if result.stdout else "Sync complete.")


### Final Step: Launch OMNI Scheduler
This cell initializes the background scheduler daemon using the `omni_cli.py` control surface.

In [ ]:
import subprocess
import os
import time

workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'
cli_path = os.path.join(workspace_path, 'omni_cli.py')

print("🚀 Launching OMNI Scheduler Daemon...")

# 1. Start the daemon
start_result = subprocess.run(['python3', 'omni_cli.py', 'start'], cwd=workspace_path, capture_output=True, text=True)
print(start_result.stdout)

# 2. Brief pause to allow for initialization
time.sleep(3)

# 3. Check status
status_result = subprocess.run(['python3', 'omni_cli.py', 'status'], cwd=workspace_path, capture_output=True, text=True)
print("--- Current OMNI Status ---")
print(status_result.stdout if status_result.stdout else "Daemon is running in background.")

### OMNI Recovery: Clearing Orphaned Locks
This cell clears stale scheduler locks and terminates colliding processes to allow a clean daemon start.

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
import os
import subprocess
import time
import signal
import sys

workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'
lock_file = os.path.join(workspace_path, '.omni_scheduler.lock')
scheduler_script = os.path.join(workspace_path, 'omnisphere_scheduler.py')

print("🔱 Initiating Direct OMNI Ignition...")

def nuclear_kill():
    """
    Enhanced process termination with granular error handling.
    """
    my_pid = os.getpid()
    my_pgid = os.getpgid(0)

    try:
        # Find processes matching 'omni' excluding the current search process
        pids = subprocess.check_output(["pgrep", "-f", "omni"], text=True).split()

        for pid_str in pids:
            pid = int(pid_str)

            # Protection: Don't kill self or members of the current process group
            if pid == my_pid or pid == my_pgid:
                continue

            try:
                os.kill(pid, signal.SIGKILL)
                print(f"  [!] Terminated PID: {pid}")
            except ProcessLookupError:
                print(f"  [?] PID {pid} already exited.")
            except PermissionError:
                print(f"  [❌] Permission denied for PID {pid}.")
            except Exception as e:
                print(f"  [⚠️] Unexpected error killing {pid}: {e}")

    except subprocess.CalledProcessError:
        print("  [i] No residual OMNI processes found.")
    except Exception as e:
        print(f"  [❌] Critical failure in process scan: {e}")

nuclear_kill()

# 2. Delete Lock and clear log
if os.path.exists(lock_file):
    os.remove(lock_file)
    print("  [!] Stale lock file purged.")

# 3. Direct Launch
print("🚀 Launching Scheduler Core...")
with open(os.path.join(workspace_path, 'logs/omnisphere_scheduler.log'), 'a') as log_out:
    subprocess.Popen(
        ['python3', 'omnisphere_scheduler.py'],
        cwd=workspace_path,
        stdout=log_out,
        stderr=log_out,
        preexec_fn=os.setpgrp
    )

time.sleep(5)

# 4. Final verification
print("--- System Verification ---")
status = subprocess.run(['python3', 'omni_cli.py', 'status'], cwd=workspace_path, capture_output=True, text=True)
print(status.stdout)

In [ ]:
import subprocess
import time
import os

workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'
max_retries = 10
retry_interval = 5

print(f"🔍 Polling OMNI Scheduler status (Max {max_retries} retries)...\n")

for i in range(1, max_retries + 1):
    result = subprocess.run(
        ['python3', 'omni_cli.py', 'status'],
        cwd=workspace_path,
        capture_output=True,
        text=True
    )

    output = result.stdout.strip()
    print(f"[Attempt {i}/{max_retries}]: {output if output else 'Starting up...'}")

    # Check for a success condition in the CLI output (e.g., 'Active' or 'Running')
    if "Active" in output or "Running" in output:
        print("\n✅ Scheduler verification successful! The system is operational.")
        break

    if i < max_retries:
        time.sleep(retry_interval)
else:
    print("\n⚠️ Timeout: Scheduler is still initializing or encountered a delay. Check logs for details.")

In [ ]:
import os

workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'
log_path = os.path.join(workspace_path, 'logs/omnisphere_scheduler.log')

print(f"📋 Inspecting OMNI Scheduler Logs: {log_path}")
print("="*60)

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        lines = f.readlines()
        # Display the last 20 lines to see recent activity/warnings
        recent_logs = lines[-20:]
        if recent_logs:
            print("".join(recent_logs))
        else:
            print("Log file is currently empty.")
else:
    print("❌ Scheduler log file not found. Check if the daemon is writing to the correct path.")

In [ ]:
import os
from IPython.display import Image, display

telemetry_path = '/content/drive/MyDrive/Gemini_Spark_Workspace/scheduler_performance.png'

if os.path.exists(telemetry_path):
    print(f"📊 Displaying OMNI Telemetry: {telemetry_path}")
    display(Image(filename=telemetry_path))
else:
    print("⚠️ Telemetry dashboard image not found yet. It may take a few minutes for the first cycle to generate the graphic.")

In [ ]:
import psutil
import os

daemon_pid = 16400

try:
    proc = psutil.Process(daemon_pid)
    with proc.oneshot():
        cpu_usage = proc.cpu_percent(interval=1.0)
        mem_info = proc.memory_info()
        mem_mb = mem_info.rss / (1024 * 1024)

    print(f"🖥️ OMNI Daemon Resource Report (PID: {daemon_pid})")
    print("="*40)
    print(f"● CPU Usage:    {cpu_usage}%")
    print(f"● Memory (RSS): {mem_mb:.2f} MB")
    print(f"● Status:       {proc.status()}")
    print(f"● Created At:   {proc.create_time()}")

except psutil.NoSuchProcess:
    print(f"❌ Process {daemon_pid} not found. The daemon might have restarted or stopped.")
except Exception as e:
    print(f"⚠️ Error retrieving metrics: {e}")

In [ ]:
import subprocess
import os

workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'

print("🕸️ Initiating OMNI Crawl & Parse Sequence...")
print("="*60)

# Using the OMNI CLI to trigger the crawler/cleaner module
# We execute the specific crawler script identified in the OMNI patterns
try:
    crawl_result = subprocess.run(
        ['python3', 'omnisphere_crawler_cleaner.py'],
        cwd=workspace_path,
        capture_output=True,
        text=True
    )

    if crawl_result.returncode == 0:
        print("✅ Crawl and Parse completed successfully.")
        print(crawl_result.stdout)
    else:
        print("❌ Crawl sequence encountered an error:")
        print(crawl_result.stderr)

except FileNotFoundError:
    print("❌ 'omnisphere_crawler_cleaner.py' not found in workspace. Verify integration.")
except Exception as e:
    print(f"⚠️ Unexpected error: {e}")

In [ ]:
import os
from pathlib import Path

workspace_path = '/content/drive/MyDrive/Gemini_Spark_Workspace'
logs_dir = os.path.join(workspace_path, 'logs')

print("📊 OMNI Crawl & Parse Results Summary")
print("="*60)

# 1. Check for specific artifacts often created by the cleaner (e.g., registry updates or specific clean data silos)
registry_path = os.path.join(workspace_path, 'linked_silos_registry.json')
if os.path.exists(registry_path):
    with open(registry_path, 'r') as f:
        import json
        registry = json.load(f)
        print(f"✅ Registry status: {len(registry.get('linked_notebooks', {}))} silos linked.")
        print(f"   Last Sync: {registry.get('last_sync', 'N/A')}")

# 2. Check for recent crawler logs
crawler_log = os.path.join(logs_dir, 'crawler_cleaner.log')
if os.path.exists(crawler_log):
    print(f"\n📝 Recent Crawler Activity (from {crawler_log}):")
    with open(crawler_log, 'r') as f:
        print("".join(f.readlines()[-15:]))
else:
    # Fallback to general watchdog if specific log is missing
    watchdog_log = os.path.join(logs_dir, 'watchdog.log')
    print(f"\n📝 Recent Watchdog/Silo Activity (from {watchdog_log}):")
    with open(watchdog_log, 'r') as f:
        print("".join(f.readlines()[-10:]))

# 3. List newly synthesized skills
skills_dir = Path(workspace_path) / 'spark_skills'
skills = list(skills_dir.glob('*.md'))
print(f"\n⚡ Total Spark Skill Manifests: {len(skills)}")
if skills:
    print("Recent manifests:")
    for s in sorted(skills, key=os.path.getmtime, reverse=True)[:5]:
        print(f" - {s.name}")

In [ ]:
import os
from pathlib import Path

skills_path = Path('/content/drive/MyDrive/Gemini_Spark_Workspace/spark_skills')

print(f"📂 Verifying Skill Manifests in: {skills_path}")
print("="*60)

if skills_path.exists():
    manifests = sorted([f.name for f in skills_path.glob('*.md')])
    print(f"✅ Total Manifests Found: {len(manifests)}")
    for i, manifest in enumerate(manifests, 1):
        print(f"  {i:02d}. {manifest}")
else:
    print("❌ Directory not found. Please ensure OMNI integration was successful.")

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

# Define workspace paths
WORKSPACE = Path('/content/drive/MyDrive/Gemini_Spark_Workspace')
REGISTRY = WORKSPACE / 'linked_silos_registry.json'
SKILLS_DIR = WORKSPACE / 'spark_skills'

print(f"--- OMNI System Diagnostics [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] ---")

# 1. Check Registry Integrity
if REGISTRY.exists():
    try:
        data = json.loads(REGISTRY.read_text())
        print(f"✅ Registry: Found ({len(data.get('linked_notebooks', {}))} notebooks linked)")
        print(f"   Last Sync: {data.get('last_sync', 'Unknown')}")
    except Exception as e:
        print(f"❌ Registry: Corrupted or unreadable - {e}")
else:
    print("⚠️ Registry: Missing (Pending first watchdog sweep)")

# 2. Skill Inventory
if SKILLS_DIR.exists():
    skills = list(SKILLS_DIR.glob('*.md'))
    print(f"✅ Skills Directory: Found ({len(skills)} manifests)")
    if skills:
        print("\nLatest 3 manifests:")
        for s in sorted(skills, key=os.path.getmtime, reverse=True)[:3]:
            mtime = datetime.fromtimestamp(s.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
            print(f" - {s.name} (Updated: {mtime})")
else:
    print("❌ Skills Directory: Not found")

# 3. Process Check (Heuristic)
import psutil
omni_procs = [p.info for p in psutil.process_iter(['pid', 'name', 'cmdline']) if 'omni' in str(p.info['cmdline']).lower()]
if omni_procs:
    print(f"\n✅ Active OMNI Processes: {len(omni_procs)}")
    for p in omni_procs[:2]:
        print(f" - PID {p['pid']}: {' '.join(p['cmdline'][:3])}...")
else:
    print("\n⚠️ Active OMNI Processes: None detected in current runtime")

In [ ]:
import psutil
import subprocess
import os
import time

WORKSPACE_PATH = "/content/drive/MyDrive/Gemini_Spark_Workspace"
os.chdir(WORKSPACE_PATH)

def check_and_restart_watchdog():
    # 1. Search for existing daemon processes
    is_running = False
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        try:
            cmdline = proc.info['cmdline']
            if cmdline and 'watchdog_silo_daemon.py' in str(cmdline):
                print(f"✅ Watchdog is currently ACTIVE (PID: {proc.info['pid']})")
                is_running = True
                break
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue

    # 2. Restart if not found
    if not is_running:
        print("⚠️ Watchdog NOT detected. Initiating restart...")

        # Kill any partial or stale processes if they exist
        try:
            daemon_process.terminate()
            time.sleep(2)
        except NameError:
            pass

        new_daemon = subprocess.Popen(
            ["python3", "watchdog_silo_daemon.py", "--interval", "180"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        print(f"🚀 Watchdog restarted successfully with PID: {new_daemon.pid}")
        return new_daemon
    return None

daemon_process = check_and_restart_watchdog()

In [ ]:
import psutil
import os

print("📡 Checking Watchdog Daemon Process...")
print("="*40)

try:
    # Check if daemon_process exists in the current kernel namespace
    if 'daemon_process' in globals() and daemon_process is not None:
        pid = daemon_process.pid
        poll = daemon_process.poll()

        if poll is None:
            print(f"✅ Status: RUNNING (PID: {pid})")
            # Fetch additional resource metrics using psutil
            proc = psutil.Process(pid)
            print(f"📊 CPU Usage: {proc.cpu_percent(interval=0.1)}%")
            print(f"📊 Memory: {proc.memory_info().rss / (1024 * 1024):.2f} MB")
        else:
            print(f"⚠️ Status: TERMINATED (Exit Code: {poll})")
    else:
        print("❌ daemon_process variable is not defined or is None.")

    # Double-check for any watchdog processes in the system
    found_pids = []
    for p in psutil.process_iter(['pid', 'cmdline']):
        if p.info['cmdline'] and 'watchdog_silo_daemon.py' in " ".join(p.info['cmdline']):
            found_pids.append(p.info['pid'])

    if found_pids:
        print(f"🔍 System search found active watchdog PIDs: {found_pids}")
    else:
        print("🔍 System search: No 'watchdog_silo_daemon.py' processes detected.")

except Exception as e:
    print(f"⚠️ Error during status check: {e}")

In [ ]:
import os
import shutil
from pathlib import Path
from IPython import get_ipython

# Target OMNI Auth File
auth_source = "/content/drive/MyDrive/OMNI/gen-lang-client-0074974088-42bdf6f9bcdc.json"
workspace_dir = Path("/content/drive/MyDrive/Gemini_Spark_Workspace")
auth_destination = workspace_dir / "auth.json"

# Ensure workspace exists
workspace_dir.mkdir(parents=True, exist_ok=True)

if os.path.exists(auth_source):
    # Synchronize file to workspace
    shutil.copy2(auth_source, auth_destination)

    # Use IPython magic to set the environment variable persistently
    auth_abs_path = str(auth_destination.absolute())
    get_ipython().run_line_magic('env', f'GOOGLE_APPLICATION_CREDENTIALS={auth_abs_path}')

    print("🛡️ OMNI Auth Integration")
    print("========================")
    print(f"✅ File Copied to: {auth_destination}")
    print(f"🔐 Credential Var (Verified): {os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')}")
    print("🚀 Verification Success: Environment is ready for authenticated API calls.")
else:
    print(f"❌ Error: Source auth file not found at {auth_source}")

In [ ]:
import os

# Explicitly set the variable using the confirmed OMNI path
auth_path = "/content/drive/MyDrive/OMNI/gen-lang-client-0074974088-42bdf6f9bcdc.json"
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = auth_path

# Verification logic
current_val = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
print("🔍 Environment Variable Verification")
print("==================================")

if current_val == auth_path:
    print(f"✅ GOOGLE_APPLICATION_CREDENTIALS set successfully to: {current_val}")
    if os.path.exists(current_val):
        print("📂 File Status: Exists and is accessible.")
        print("🚀 System is ready for authenticated Gemini/Google Cloud operations.")
    else:
        print("❌ File Status: Path is set, but the file was not found. Please verify Drive is mounted.")
else:
    print(f"❌ Failed to set variable. Current value: {current_val}")